In [1]:
!pip install nbstripout
!nbstripout your_notebook.ipynb

Could not strip 'your_notebook.ipynb': file not found


In [2]:
!pip install --pre -U langchain langchain-openai langchain_community langchain_core langchain_text_splitters  unstructured langchain_huggingface langchain_cohere

  Using cached langchain_cohere-0.5.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached types_pyyaml-6.0.12.20260408-py3-none-any.whl.metadata (1.7 kB)
  Using cached fastavro-1.12.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.8 kB)
  Using cached types_requests-2.33.0.20260408-py3-none-any.whl.metadata (2.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.1 MB/s eta 0:00:00


## the model
we will use local model as a judge
we will use "Qwen/Qwen2.5-7B-Instruct"


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LlamaTokenizer ,LlamaForCausalLM ,GenerationConfig ,pipeline ,AutoTokenizer ,AutoModelForCausalLM
import torch
from langchain_huggingface  import HuggingFacePipeline,ChatHuggingFace

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Chatbot Evaluation

In [5]:
from google.colab import userdata
import os
langsmith_api_key = userdata.get('LANGSMITH')

os.environ['LANGCHAIN_TRACING'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = langsmith_api_key


### Define dataset

In [ ]:
from langsmith import Client

client = Client()

# define :database : the test case

database_name = "Chatbot evaluation"

dataset = client.create_dataset(
        dataset_name=database_name,
        description="Chatbot evaluation"
                                )

In [ ]:
dataset.id

UUID('7e0c30d3-8226-4540-89ce-535be3188c5a')

In [ ]:
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs":{"question":"what is langchain"},
            "outputs":{"answer":"A framework for building LLM applications"}
        },
        {
            "inputs":{"question":"What is LangSmith?"},
            "outputs":{"answer":"A platform for observing and evaluating LLM applications"}
        },
        {
            "inputs":{"question":"What is OpenAI?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs":{"question":"What is Google?"},
            "outputs":{"answer":"A technology company known for search"}
        },

        {
            "inputs":{"question":"What is Mistral?"},
            "outputs":{"answer":"A company that creates Large Language Models"}
        },
        {
            "inputs": {"question": "What is Hugging Face?"},
            "outputs": {"answer": "A company that provides tools and models for natural language processing"}
        },
        {
            "inputs": {"question": "What is TensorFlow?"},
            "outputs": {"answer": "An open-source machine learning framework developed by Google"}
        },
        {
            "inputs": {"question": "What is PyTorch?"},
            "outputs": {"answer": "An open-source machine learning library used for deep learning applications"}
        },
        {
            "inputs": {"question": "What is Anthropic?"},
            "outputs": {"answer": "A company that develops AI systems and large language models"}
        },
        {
            "inputs": {"question": "What is Cohere?"},
            "outputs": {"answer": "A company that provides language AI models and APIs for developers"}
        }

    ]
)

{'example_ids': ['fa40eca6-a285-479c-8c78-ed1294c6f4bf',
  'fab71260-88a8-4870-98dc-eaf85c4df54d',
  '519e3d5f-d217-496c-8c8a-3621c7e62d9c',
  '244f6046-e586-4170-83cb-9502a918c94c',
  '8d39afd7-23e3-459c-810d-3260a87c998f',
  'b3cea7e3-806f-4db7-be6b-16443b35641f',
  '6ba44ec9-d672-476e-8498-9becc0e751bc',
  'e2e08821-b462-4635-a250-79be37bd48cc',
  '60cf18ee-931f-49bd-b002-d39630f43194',
  '0e93fd35-8004-461e-b849-fad9323506c4'],
 'count': 10,
 'as_of': '2026-04-29T16:11:07.655766603Z'}

### Define Metrics (LLM As A Judge)
Qwen/Qwen2.5-7B-Instruct

In [ ]:

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading a student's answer.
          Question: {inputs['question']}
          Correct Answer: {reference_outputs['answer']}
          Student Answer: {outputs['response']}

          Instructions:
          - Respond with ONLY one word.
          - Allowed answers: CORRECT or INCORRECT
          - Do not explain.

          grade:"""

      messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]

      text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
      )

      inputs = tokenizer([text], return_tensors="pt").to(model.device)

      outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1
        )

      generated_ids = outputs[0][inputs.input_ids.shape[1]:]

      response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
      )



      return response.strip() == "CORRECT"

In [ ]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs:dict, reference_outputs:dict) -> bool:
    response = outputs.get('response', "")
    return int(len(response) < 2 * len(reference_outputs['answer']))

### Run Evaluations

In [ ]:
defult_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str ,instructions:str = defult_instructions) -> str:
  messages=[
           {"role": "system", "content": instructions},
           {"role": "user", "content": question}
       ]

  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
  )

  inputs = tokenizer([text], return_tensors="pt").to(model.device)

  outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1
    )

  generated_ids = outputs[0][inputs.input_ids.shape[1]:]

  response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
  )

  return response.strip()

In [ ]:
### Call my_app for every datapoints

def ls_target(inputs):
    try:
        result = my_app(inputs['question'])
        return {"response": result}
    except Exception as e:
        return {"response": "", "error": str(e)}


In [ ]:
## Run our evaluation
experiment_results = client.evaluate(
    ls_target, ## Your AI system
    data = database_name,
    evaluators = [correctness,concision],
    experiment_prefix="Qwen/Qwen2.5-7B-Instruct"

)

View the evaluation results for experiment: 'Qwen/Qwen2.5-7B-Instruct-679a52a9' at:
https://smith.langchain.com/o/d43acb99-e975-4cc0-abe8-e96f5723bc0a/datasets/7e0c30d3-8226-4540-89ce-535be3188c5a/compare?selectedSessions=ed8d12e5-fdd7-470c-85d5-05cb7e5a7d7f




0it [00:00, ?it/s]